In [1]:
import pandas as pd

# Search Evaluation

We use the ground truth to evaluate our search engine because, again, if our search only returns the most relevant results, then our RAG will provide better output. 

In [2]:
df_ground_truth = pd.read_csv("data/ground_truth-new.csv")

df_ground_truth.head()

,question,document
0,Can I still join the course if I found it late?,74eb249bbf
1,Is it too late to start this course now?,74eb249bbf
2,"If I join after the course started, can I stil...",74eb249bbf
3,What do I need to do to be eligible for the ce...,74eb249bbf
4,Do I have to submit the project before submiss...,74eb249bbf


In [3]:
ground_truth = df_ground_truth.to_dict(orient="records")
ground_truth[:5]

[{'question': 'Can I still join the course if I found it late?',
  'document': '74eb249bbf'},
 {'question': 'Is it too late to start this course now?',
  'document': '74eb249bbf'},
 {'question': 'If I join after the course started, can I still get a certificate?',
  'document': '74eb249bbf'},
 {'question': 'What do I need to do to be eligible for the certificate if I join late?',
  'document': '74eb249bbf'},
 {'question': 'Do I have to submit the project before submissions close to get certified?',
  'document': '74eb249bbf'}]

Now we download the documents (aka, our knowledge base) and index them.

In [4]:
from utils.ingest import load_faq_data, build_index

raw_documents = load_faq_data()

documents_llm = [document for document in raw_documents if document["course"] == "llm-zoomcamp"]

index = build_index(documents_llm)

Now we create the text search; vector search will come later. 

In [5]:
def text_search(query:str):
    boost_dict = {"question":3.0, "section":0.5}

    results = index.search(query, num_results=5, boost_dict=boost_dict)

    return results

We'll use the question below for evaluation.

In [6]:
q = ground_truth[0]
q

{'question': 'Can I still join the course if I found it late?',
 'document': '74eb249bbf'}

Let's run a search for this question

In [7]:
doc_id = q['document']
results = text_search(query=q['question'])

We'll compare the retrieved document IDs with the correct document ID: 

In [8]:
for result in results: 
    print(f'{result["id"]}=={doc_id}: {result["id"]==doc_id}')

74eb249bbf==74eb249bbf: True
a9353fadfe==74eb249bbf: False
9f689c185f==74eb249bbf: False
977bf7786c==74eb249bbf: False
69d122f12e==74eb249bbf: False


So, 1 in 5 of our results were relevant. We can save these results as a list for future evaluation. 

In [9]:
relevance = [int(result["id"] == doc_id) for result in results]

In [10]:
relevance

[1, 0, 0, 0, 0]

Wrap it all in a function to make it reusable. 

In [11]:
def compute_relevance_text(q):
    doc_id = q['document']
    results = text_search(query=q['question'])

    relevance = [int(result["id"] == doc_id) for result in results]

    return relevance


In [12]:
q = ground_truth[11]
print(q["question"])
compute_relevance_text(q)
# [1, 0, 0, 0, 0]

Where can I find the livestream link for office hours or workshop sessions before they start?


[1, 0, 0, 0, 0]

Now we do the same thing for all of our ground truth answers.

In [13]:
from tqdm.auto import tqdm

def compute_relevance_total_text(ground_truth):
    
    relevance_total = [compute_relevance_text(question) for question in ground_truth]

    return relevance_total

/Users/evan/Projects/llm-zoomcamp-2026-code/module_04_evaluation/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


What is the result for the first 15 questions? 

In [14]:
ground_truth_sample = ground_truth[:15]

In [15]:
relevance_total_text = compute_relevance_total_text(ground_truth_sample)

In [16]:
# Inspect the results 
relevance_total_text

[[1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 1, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0]]

In [17]:
# Calculate the hit rate
total = 0 

for relevance in relevance_total_text:
    total += sum(relevance)

round(total/(len(relevance_total_text)) *100, 1)

80.0

We're going to want to calculate the relevance for both the text search as well as the vector search. 

As such, we can generalize the function.

In [18]:
from collections.abc import Callable


def compute_relevance(q:dict, searchfunction:Callable) -> list: 
    doc_id = q["document"]
    results = searchfunction(query=q["question"])

    relevance = [int(d["id"]==doc_id) for d in results]

    return relevance

Now we expand it out to compute the total relevance. 

In [19]:
def compute_relevance_total(ground_truth:list, search_function:Callable):

    relevance_total = [compute_relevance(q, search_function)  for q in ground_truth]

    return relevance_total

Test run on the sample

In [20]:
sample_set = compute_relevance_total(ground_truth=ground_truth_sample, search_function=text_search)

Success! Let's try it on all of them :D 

In [21]:
relevance_text_search = compute_relevance_total(ground_truth=ground_truth, search_function=text_search)

# Metrics

We can quantify how *good* our search function is by identify our hit rate. 

In [22]:
def calculate_hit_rate(relevance:list):
    # Calculate the hit rate
    total = 0 
    
    for line in relevance:
        total += sum(line)
    
    return round(total/(len(relevance)) *100, 1)

In [23]:
print(f'The hit rate for the sample set is {calculate_hit_rate(sample_set)}%.')
print(f'The hit rate for the entire set is {calculate_hit_rate(relevance_text_search)}%.')

The hit rate for the sample set is 80.0%.
The hit rate for the entire set is 85.4%.


# Mean Reciprocal Rank (MRR)

What if we want to know the order the correct document was found in? 

In that case, we look at the mean reciprocal rank (MRR): the lower the index, the better the result. 

For each query, the score is based on the rank of the first correct document:

- position 1: score is 1.0
- position 2: score is 0.5
- position 3: score is 0.333
- not found: score is 0


For example, the score for `sample_set[0]` is `1` because the index of the correct document is `0`:

In [24]:
sample_set

[[1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 1, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0]]

In [25]:
def calculate_mrr(relevance):

    mrr_score = 0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                mrr_score = mrr_score +1/(rank+1)
                break

    return mrr_score / len(relevance)

In [26]:
calculate_mrr(sample_set)

0.6833333333333333

In [27]:
example = [
    [1, 0, 0, 0, 0],
    [0, 1, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [0, 0, 0, 0, 0],
    [0, 1, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [0, 0, 1, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
]

In [28]:
calculate_mrr(example)

0.8222222222222222

In [29]:
calculate_hit_rate(example)

93.3

In [30]:
def evaluate(ground_truth:list, search_function:Callable):
    relevance_total = compute_relevance_total(ground_truth, search_function)

    hit_rate = calculate_hit_rate(relevance_total)
    mrr = calculate_mrr(relevance_total)

    return {
        "hit_rate": hit_rate,
        'mrr': mrr
    }


In [31]:
evaluate(ground_truth, text_search)

{'hit_rate': 85.4, 'mrr': 0.7322977346278313}

# Search Parameter Tuning 

The documents we're searching over have three areas of interest: 
- course
- section
- question


We've already filtered out all questions that aren't about llm-zoomcamp, but we can index higher or lower on the `section` and `question` to see if we can return better results. 

In [32]:
documents_llm[0]

{'id': '74eb249bbf',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

To refresh our memory, here is our `text_search` function: 

In [33]:
def text_search(query:str):
    boost_dict = {"question":3.0, "section":0.5}

    results = index.search(query, num_results=5, boost_dict=boost_dict)

    return results

Let's modify it to try different weights for the `question`. 

In [34]:
def search_boost(query, question_boost:list):
    boost_dict = {'question': question_boost, 'section':0.5}

    boosted_results = index.search(query, num_results=5, boost_dict=boost_dict)

    return boosted_results

In [35]:
weights = [0.5, 1.0, 3.0, 5.0, 10.0]

for boost in weights:
    result = evaluate(
        ground_truth,
        lambda query, boost=boost: search_boost(query, boost)
    )
    print(f"boost={boost}: {result}")

boost=0.5: {'hit_rate': 88.0, 'mrr': 0.7838834951456308}
boost=1.0: {'hit_rate': 89.7, 'mrr': 0.78378640776699}
boost=3.0: {'hit_rate': 85.4, 'mrr': 0.7322977346278313}
boost=5.0: {'hit_rate': 82.7, 'mrr': 0.6992556634304203}
boost=10.0: {'hit_rate': 78.8, 'mrr': 0.6674433656957927}


Let's tune everything and see what happens: 

In [36]:
def search_boosts(query, question_boost:list, answer_boost:list, section_boost:list):
    boost_dict = {'question': question_boost, 'section':section_boost, 'answer': answer_boost}

    boosted_results = index.search(query, num_results=5, boost_dict=boost_dict)

    return boosted_results

Do a grid search: 

In [37]:
results = []

question_weights = [1.0, 2.0, 5.0]
answer_weights = [1.0, 2.0, 4.0, 10.0]
section_weights =  [0.1, 0.2, 0.5]

for question_boost in question_weights:
    for answer_boost in answer_weights:
        for section_boost in section_weights:
            print(
                f"Evaluating question_boost={question_boost},"
                f" answer_boost={answer_boost},"
                f" section_boost={section_boost}..."
            )
            result = evaluate(
                ground_truth,
                lambda query, question_boost=question_boost, answer_boost=answer_boost, section_boost=section_boost: search_boosts(
                    query, 
                    question_boost, 
                    answer_boost, 
                    section_boost
                )
            )

            results.append({
                "question": question_boost,
                'answer': answer_boost,
                "section": section_boost,
                "hit_rate": result["hit_rate"],
                "mrr": result["mrr"]
            })

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.1...
Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.2...
Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.5...
Evaluating question_boost=1.0, answer_boost=2.0, section_boost=0.1...
Evaluating question_boost=1.0, answer_boost=2.0, section_boost=0.2...
Evaluating question_boost=1.0, answer_boost=2.0, section_boost=0.5...
Evaluating question_boost=1.0, answer_boost=4.0, section_boost=0.1...
Evaluating question_boost=1.0, answer_boost=4.0, section_boost=0.2...
Evaluating question_boost=1.0, answer_boost=4.0, section_boost=0.5...
Evaluating question_boost=1.0, answer_boost=10.0, section_boost=0.1...
Evaluating question_boost=1.0, answer_boost=10.0, section_boost=0.2...
Evaluating question_boost=1.0, answer_boost=10.0, section_boost=0.5...
Evaluating question_boost=2.0, answer_boost=1.0, section_boost=0.1...
Evaluating question_boost=2.0, answer_boost=1.0, section_boost=0.2...
Evaluating questi

In [38]:
df_results = pd.DataFrame(results)
df_results.sort_values("mrr", ascending=False).head(10)

,question,answer,section,hit_rate,mrr
7,1.0,4.0,0.2,96.9,0.875405
4,1.0,2.0,0.2,96.7,0.873107
35,5.0,10.0,0.5,97.3,0.871133
3,1.0,2.0,0.1,97.3,0.871133
19,2.0,4.0,0.2,97.3,0.871133
6,1.0,4.0,0.1,96.7,0.870939
33,5.0,10.0,0.1,96.9,0.870097
18,2.0,4.0,0.1,97.3,0.870065
20,2.0,4.0,0.5,96.3,0.869741
34,5.0,10.0,0.2,96.9,0.869288


Four of our Top 5 results have the approximate ratio:  
- question: 1
- answer: 2 
- section: .2


So, let's plug those values into our search: 

In [39]:
def text_search(query):
    boost_dict = {'question': 1.0, 'answer': 2.0, 'section': 0.2}

    results = index.search(query, num_results=5, boost_dict=boost_dict)

    return results

# RAG and Agent Evaluation

We've evaluated the search function, the next step is to evaluate what the agent produces. 

We also need to inspect the: 
- prompt: are we providing enough/the correct context for the agent to work properly? 
- llm: is the llm ignoring the context? 